<a href="https://colab.research.google.com/github/Ayomide-Fagbolade/tabular_reinforcement_learning/blob/main/tabular_reinforcement_learning%20/notebooks/Tabular_RL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# My Personal RL Notebook: What RL Is

**Reinforcement Learning (RL)** is learning what to do by trial and error, driven by a reward signal. Nobody tells the agent the correct action. It tries different actions, receives numerical feedback, and learns which behaviors yield higher rewards over time. This trial-and-error approach separates RL from supervised learning, where each training example is paired with an explicit ground-truth answer.

---

### Key Components

* **State ($s$):** What the situation is right now.
* **Action ($a$):** What the agent chooses to do.
* **Reward ($r$):** A numerical value returned by the environment after each action, indicating how good that step was.
* **Policy ($\pi$):** The agent's rule for picking an action in each state. This is the main function or rule we aim to optimize.

---

### Core Objective

The ultimate goal is **not** to maximize the immediate reward on the next step. Instead, it is to maximize the **cumulative reward** over the entire trajectory or episode. Sometimes, an agent must accept a smaller (or negative) reward in the short term to reach a significantly higher reward later.


### Tiny World

The world is a corridor of 4 cells: **0, 1, 2, 3**.

* **Starting Position:** Cell 0
* **Goal:** Reaching cell 3 yields a reward of **+1** and terminates the episode.
* **Actions:**
* Left (`0`)
* Right (`1`)


* **Discount Factor ($\gamma$):** $0.9$ (encourages reaching the goal as quickly as possible)

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def step(s, a):                          # the environment
    s_new = min(s + 1, 3) if a == 1 else max(s - 1, 0)
    reward = 1.0 if s_new == 3 else 0.0
    done = (s_new == 3)
    return s_new, reward, done

def run_episode(policy, gamma=0.9):      # the loop
    s, done, G, t = 0, False, 0.0, 0
    while not done:
        a = policy(s)
        s, r, done = step(s, a)
        G += gamma**t * r                # discounted return
        t += 1
    return G

always_right = lambda s: 1
random_policy = lambda s: rng.integers(0, 2)

In [ ]:
print("always_right:", run_episode(always_right))
rets = [run_episode(random_policy) for _ in range(2000)]
print("random avg  :", np.mean(rets))

always_right: 0.81
random avg  : 0.43373936168169464


**Step 3: the value of a state**

**Core theory**

Last step, the return was the score of one episode, and it varied from run to run for the random policy. The **value** of a state, `V(s)`, is the *average* return over many episodes that start in `s` and follow a policy.

This gives you two ways to get it:

1. **Sample it.** Run many episodes from `s` and average the returns. This works with no knowledge of how the world is built.
2. **Compute it by recursion.** The first step gives a reward `r` and lands you in `s'`, and from there the value is `V(s')`, shrunk by `γ`. So `V(s) = r + γ·V(s')`. For a random policy you also average over the choices it might make.

**Supporting code**

Replace your `run_episode` with this version, which takes a starting cell. Keep `step`, `always_right`, and `random_policy` as they are:


In [ ]:


def run_episode(policy, start=0, gamma=0.9):
    s, done, G, t = start, False, 0.0, 0
    while not done:
        s, r, done = step(s, policy(s))
        G += gamma**t * r
        t += 1
    return G

def V_estimate(policy, n=5000):
    return [np.mean([run_episode(policy, start) for _ in range(n)]) for start in range(3)]

print("always_right:", np.round(V_estimate(always_right), 3))
print("random      :", np.round(V_estimate(random_policy), 3))




always_right: [0.81 0.9  1.  ]
random      : [0.433 0.523 0.74 ]


The `always_right` numbers are exactly the recursion at work. From cell 2, one step gives +1, so `V(2) = 1`. From cell 1, you get 0 now and then land in cell 2, so `V(1) = 0 + 0.9 × 1 = 0.9`. From cell 0, `V(0) = 0 + 0.9 × 0.9 = 0.81`. The value falls by a factor of γ for each step further from the goal.

The random policy's values are lower everywhere because it wastes steps. The exact values, from solving the recursion, are `[0.429, 0.524, 0.736]`, so your sampled numbers should be within about 0.01 of those.

**Check before the next step:** run it and tell me what your two lines printed. Then answer this: why is the random policy's `V(2)` (about 0.73) so much lower than `always_right`'s `V(2)` (exactly 1.0), when both are only one cell away from the goal?

its because  it doesnt always go to s3 from s2

Right. From cell 2 the random policy only steps right half the time. The other half it steps left to cell 1, and it then has to come back, so the +1 arrives later and is discounted more.

You can check this with numbers. Half the time you get +1 immediately. The other half you get 0 now and land in cell 1, whose value is about 0.523, shrunk by γ:

```
V(2) = 0.5 · (1)  +  0.5 · (0 + 0.9 · 0.523)  ≈ 0.5 + 0.235 = 0.735
```

That matches the 0.733 from sampling. You've just used the **Bellman equation** in words, so next we'll write it out properly and compute values without sampling any episodes. I'll run the code first so the numbers I quote are right.**Step 4: the Bellman equation**

**Core theory**

Your answer already contains the idea. The value of a state is an average over what the policy might do, where each option contributes its reward plus the discounted value of where it lands:

```
V(s) = Σ_a  π(a|s) · [ r(s,a) + γ · V(s') ]
```

Here `π(a|s)` is the probability that the policy picks action `a` in state `s`. For the random policy it is 0.5 for each action. For `always_right` it is 1 for right and 0 for left, which is why that policy's value came out as a simple chain.

`V` appears on both sides, so you can't read the answer off directly. The trick is to guess `V = 0` everywhere, apply the equation to get a slightly better guess, and repeat. It converges, and the goal cell's value stays at 0 because the episode ends there. In the sweep output above, information about the +1 spreads backward from the goal one cell per sweep.



In [ ]:

gamma = 0.9
V = np.zeros(4)                        # V[3] is the goal cell, stays 0
for sweep in range(1, 101):
    V_new = np.zeros(4)
    for s in range(3):
        for a in (0, 1):
            s_new, r, done = step(s, a)
            V_new[s] += 0.5 * (r + gamma * V[s_new])   # 0.5 = chance the random policy picks a
    V = V_new
    if sweep <= 3 or sweep == 100:
        print(f"sweep {sweep:3d}:", np.round(V[:3], 3))


sweep   1: [0.  0.  0.5]
sweep   2: [0.    0.225 0.5  ]
sweep   3: [0.101 0.225 0.601]
sweep 100: [0.429 0.524 0.736]


<details>
<summary><b>Click to expand: Step-by-step Bellman Sweep Breakdown</b></summary>

### Initial State
At $t=0$, we initialize $V(s) = 0$ for all states ($[0, 0, 0, 0]$).

---

### Sweep 1: Information enters at the goal boundary
We compute $V_{new}[s] = 0.5 \cdot \sum_a (r + \gamma \cdot V[s_{new}])$:

* **Cell 0:** Left $\rightarrow 0$ ($r=0, V[0]=0$), Right $\rightarrow 1$ ($r=0, V[1]=0$).  
  $$V[0] = 0.5(0) + 0.5(0) = \mathbf{0.0}$$
* **Cell 1:** Left $\rightarrow 0$ ($r=0, V[0]=0$), Right $\rightarrow 2$ ($r=0, V[2]=0$).  
  $$V[1] = 0.5(0) + 0.5(0) = \mathbf{0.0}$$
* **Cell 2:** Left $\rightarrow 1$ ($r=0, V[1]=0$), Right $\rightarrow 3$ (**Goal!** $r=1.0, V[3]=0$).  
  $$V[2] = 0.5(0 + 0.9 \cdot 0) + 0.5(1.0 + 0.9 \cdot 0) = \mathbf{0.5}$$

**Result:** `[0.0, 0.0, 0.5]`  
*Only Cell 2 gets a value because it is the only cell directly adjacent to the $+1.0$ goal reward.*

---

### Sweep 2: Value propagates to Cell 1
Now $V = [0.0, 0.0, 0.5, 0.0]$:

* **Cell 0:**  
  $$V[0] = 0.5(0 + 0.9 \cdot 0) + 0.5(0 + 0.9 \cdot 0) = \mathbf{0.0}$$
* **Cell 1:** Stepping right lands in Cell 2, which now has $V[2] = 0.5$:  
  $$V[1] = 0.5(0 + 0.9 \cdot 0) + 0.5(0 + 0.9 \cdot 0.5) = 0.5(0.45) = \mathbf{0.225}$$
* **Cell 2:**  
  $$V[2] = 0.5(0 + 0.9 \cdot 0) + 0.5(1.0 + 0.9 \cdot 0) = \mathbf{0.5}$$

**Result:** `[0.0, 0.225, 0.5]`  
*Cell 1 learns that moving right leads to a state (Cell 2) with future value.*

---

### Sweep 3: Value reaches Cell 0 & feedback loop starts
Now $V = [0.0, 0.225, 0.5, 0.0]$:

* **Cell 0:** Stepping right lands in Cell 1 ($V[1] = 0.225$):  
  $$V[0] = 0.5(0 + 0.9 \cdot 0) + 0.5(0 + 0.9 \cdot 0.225) = \mathbf{0.101}$$
* **Cell 1:** Stepping left lands in Cell 0 ($V[0]=0$), right lands in Cell 2 ($V[2]=0.5$):  
  $$V[1] = 0.5(0 + 0.9 \cdot 0) + 0.5(0 + 0.9 \cdot 0.5) = \mathbf{0.225}$$
* **Cell 2:** Stepping left lands in Cell 1 ($V[1] = 0.225$), right lands in Goal ($r=1.0$):  
  $$V[2] = 0.5(0 + 0.9 \cdot 0.225) + 0.5(1.0 + 0.9 \cdot 0) = 0.10125 + 0.5 = \mathbf{0.601}$$

**Result:** `[0.101, 0.225, 0.601]`  
*The reward signal has now reached all cells in the grid, and Cell 2 increases as Cell 1's value feeds back into it.*

---

### Sweep 100: Convergence ($V^*$)
After repeating this process, the values stabilize completely (reach equilibrium where $V_{new} \approx V$):

$$\begin{aligned} V(0) &= \mathbf{0.429} \\ V(1) &= \mathbf{0.524} \\ V(2) &= \mathbf{0.736} \end{aligned}$$

**Result:** `[0.429, 0.524, 0.736]`  
*These exact numbers match our earlier 5,000-episode Monte Carlo simulation without needing to run a single random episode.*

</details>



The final line matches the sampled estimates from step 3 (0.43, 0.523, 0.733). This time no episodes were simulated. The recursion produced the same values from the rules of the world alone, and that is the core of planning.

To see how each policy changes the equation, replace `0.5 * (...)` with an `if a == 1:` that adds the full `(r + gamma * V[s_new])`. That is `always_right`, and after a few sweeps it should give `[0.81, 0.9, 1.0]`.

**Check before the next step:** look at sweep 1, where only cell 2 has a nonzero value (0.5). Why is that the only cell that gets anything on the first pass? Run the code, tell me if your numbers match, and answer that.

**Step 5: from loops to arrays**

**Core theory**

Last step's code looped over every state and action and called `step`. That works only when you can call the world like a function. The notebook instead hands you the world's rules as two lookup tables:

- `T[s, a, s']`: the probability that taking action `a` in state `s` lands you in `s'`.
- `R[s, a, s']`: the reward for that move.

The Bellman equation is unchanged. You have to pick out the right slice of the tables for your policy. A deterministic policy is an array `pi` with one action per state (`pi[2] = 1` means "in cell 2, go right"). `T[range(S), pi]` takes, for each state `s`, the row for the action `pi[s]`. That collapses `T` from shape `(s, a, s')` to `(s, s')`, which is "where do I go if I follow this policy". Doing the same to `R` and multiplying the two gives the loop's sum over `s'` in one line. In this notebook policies are deterministic, so there's no sum over actions yet. The random policy from step 4 is stochastic, so we'll skip it here.





**What you should see**

```
(4, 2, 4) (4, 4)
[0. 0. 0. 1.]
always_right: [0.81 0.9  1.   0.  ]
always_left : [0. 0. 0. 0.]
```

- Line 1 shows the slicing collapsing the action axis, from `(4, 2, 4)` to `(4, 4)`.
- Line 2 is row 2 of that table: from cell 2, going right, you land in cell 3 with probability 1.
- Line 3 is the same `[0.81, 0.9, 1.0]` as before, so the array version agrees with the loop version.
- Line 4 says a policy that never reaches the goal is worth 0.

`evaluate` is the notebook's `policy_eval_numerical` with the stopping test left out. That test (stop once `V` changes by less than a tiny `eps`) is a few extra lines, and it's the next thing we'll add.

**Check before the next step:** in `evaluate`, `V` has shape `(4,)`, while `Rp` has shape `(4, 4)`. Why can `Rp + gamma * V` be added without error, and does `V` line up with the rows or the columns of `Rp`? Think about what each column of `Rp` stands for, and answer that one first, since it's the broadcasting rule that causes most of the bugs in this section.

In [ ]:
S, A = 4, 2
T = np.zeros((S, A, S)); R = np.zeros((S, A, S))
for s in range(3):
    for a in range(A):
        s_new, r, done = step(s, a)
        T[s, a, s_new] = 1
        R[s, a, s_new] = r
T[3, :, 3] = 1

def evaluate(pi, gamma=0.9, sweeps=100):
    V = np.zeros(S)
    P  = T[range(S), pi]          # (s, s')
    Rp = R[range(S), pi]          # (s, s')
    for _ in range(sweeps):
        V = (P * (Rp + gamma * V)).sum(1)
    return V

pi_right = np.array([1, 1, 1, 1])
print(T.shape, T[range(S), pi_right].shape)
print(T[range(S), pi_right][2])
print("always_right:", evaluate(pi_right).round(3))
print("always_left :", evaluate(np.array([0, 0, 0, 0])).round(3))

(4, 2, 4) (4, 4)
[0. 0. 0. 1.]
always_right: [0.81 0.9  1.   0.  ]
always_left : [0. 0. 0. 0.]


Key Takeaway: Everything we have covered so far falls squarely under Dynamic Programming (DP)—a family of model-based planning algorithms (like Policy Evaluation) that compute exact value functions by sweeping over complete environment dynamics ($T$ and $R$) without running real-time sampled episodes.

Step 6: two ways to evaluate a policy, and when to stop

Core theory

You can compute V in two ways, and the notebook has an exercise for each:

Iterate the Bellman update until V stops changing. Stop when the biggest change in any state, max|V_new − V_old|, falls below a tiny eps (or after a fixed maximum number of sweeps, in case it never settles).
Solve directly. Write the Bellman equation for all states at once as v = r + γ·P·v, where P is the (s, s') matrix you already built and r is the expected immediate reward of each state, r[s] = Σ_s' P[s,s'] · Rp[s,s']. Moving v to one side gives (I − γP)·v = r, which is a linear system a solver can handle in one step.

In [ ]:
def eval_numerical(pi, gamma=0.9, eps=1e-8, max_iter=10_000):
    P, Rp = T[range(S), pi], R[range(S), pi]
    V = np.zeros(S)
    for i in range(max_iter):
        V_new = (P * (Rp + gamma * V)).sum(1)
        if np.abs(V_new - V).max() < eps:
            print("converged after", i + 1, "sweeps")
            return V_new
        V = V_new
    print("did not converge")
    return V

def eval_exact(pi, gamma=0.9):
    P, Rp = T[range(S), pi], R[range(S), pi]
    r = (P * Rp).sum(1)                      # expected immediate reward, shape (s,)
    return np.linalg.solve(np.eye(S) - gamma * P, r)

pi_right = np.array([1, 1, 1, 1])
print(eval_numerical(pi_right).round(3))
print(eval_exact(pi_right).round(3))

converged after 4 sweeps
[0.81 0.9  1.   0.  ]
[0.81 0.9  1.   0.  ]


**Step 7: Q-values and policy improvement**

**Core theory**

So far `V(s)` has scored a state under a policy. The **Q-value** `Q(s, a)` scores one specific first move: take action `a` now, then follow the policy afterwards.

```
Q(s,a) = Σ_s'  T(s'|s,a) · [ R(s,a,s') + γ·V(s') ]
```

This is the Bellman right-hand side with the action left free instead of fixed by the policy. `V(s)` is just `Q(s, π(s))`, the Q-value of the action the policy actually takes.

That gives you an improvement rule. If some other action `a'` has `Q(s, a') > V(s)`, switching to `a'` in state `s` beats the current policy there. The rule applied to every state at once is:

```
π_new(s) = argmax_a Q(s, a)
```

This is greedy improvement, and it never makes the policy worse.

In [ ]:



def improve(V, gamma=0.9):
    Q = (T * (R + gamma * V)).sum(-1)      # (s, a)
    return Q, Q.argmax(1)

pi = np.array([0, 0, 0, 0])                # always left
V = eval_exact(pi)
Q, pi_new = improve(V)
print(Q.round(3))
print("new policy:", pi_new)
print("its value :", eval_exact(pi_new).round(3))


[[0. 0.]
 [0. 0.]
 [0. 1.]
 [0. 0.]]
new policy: [0 0 1 0]
its value : [-0.  0.  1.  0.]


```

The `improve` line is the same pattern as evaluation, except `T` and `R` keep all their actions. `R + gamma * V` broadcasts `V` over the last axis (`s'`), just as you checked, and `.sum(-1)` collapses `s'`, leaving shape `(s, a)`.

**What you should see**

```
[[0. 0.]
 [0. 0.]
 [0. 1.]
 [0. 0.]]
new policy: [0 0 1 0]
its value : [-0.  0.  1.  0.]
```

Cell 2 is the only one with a nonzero Q-value, and only for action 1, right, because that's the only move that earns a reward right away. That is the same reason only cell 2 lit up on sweep 1 in step 4. The other cells have all-zero Q-values, so `argmax` breaks the tie by picking the first action (left). The new policy is "go right from cell 2, otherwise do what you did before", and it's already better: its value at cell 2 went from 0 to 1.

**Check before the next step:** improvement is one round of a loop. Predict what the next round will change: if you now evaluate `pi_new` and improve again, which cell's action will flip, and why that cell? Then run these lines to check yourself:



In [ ]:

V = eval_exact(pi_new)
Q, pi_newer = improve(V)
print(Q.round(3))
print("newer policy:", pi_newer)



[[0.  0. ]
 [0.  0.9]
 [0.  1. ]
 [0.  0. ]]
newer policy: [0 1 1 0]


Policy Evaluation vs. Policy ImprovementPolicy Evaluation: Fixes $\pi$ and computes $V^\pi(s)$—measuring how well the current policy performs across all states.Policy Improvement: Computes $Q(s, a)$ to test every available action $a$ for one step, followed by $V^\pi(s')$ thereafter. If an alternative action improves the return, we update $\pi(s) \leftarrow \arg\max_a Q(s, a)$.By alternating between evaluating $V^\pi(s)$ and greedily updating $\pi(s)$ via $Q(s, a)$, we get Policy Iteration, which is guaranteed to converge to the optimal policy $\pi^*$.

Step 8: find_optimal_policy

Core theory

You now have both halves of policy iteration:

Evaluate: given a policy, compute its value V (step 6).
Improve: given V, pick the greedy action in every state (step 7).

Alternate them: π₀ → V₀ → π₁ → V₁ → π₂ → .... Each round the policy is at least as good as the one before, and strictly better unless it can't be improved. There are only finitely many deterministic policies, so the loop has to stop somewhere. It stops when the improvement step returns the policy you already had. At that point no single action switch helps anywhere, which means V satisfies the Bellman equation with a max over actions, and that is the definition of an optimal policy.

In [ ]:
def eval_exact(T, R, pi, gamma):
    S = T.shape[0]
    P, Rp = T[range(S), pi], R[range(S), pi]
    r = (P * Rp).sum(1)
    return np.linalg.solve(np.eye(S) - gamma * P, r)

def improve(T, R, V, gamma):
    return (T * (R + gamma * V)).sum(-1).argmax(1)

def find_optimal(T, R, gamma=0.9, max_iter=10_000):
    pi = np.zeros(T.shape[0], dtype=int)         # any starting policy works
    for _ in range(max_iter):
        V = eval_exact(T, R, pi, gamma)
        pi_new = improve(T, R, V, gamma)
        if np.array_equal(pi_new, pi):
            return pi
        pi = pi_new
    print("did not converge")
    return pi

pi = find_optimal(T, R)
print("corridor:", pi, eval_exact(T, R, pi, 0.9).round(3))

corridor: [1 1 1 0] [0.81 0.9  1.   0.  ]


**Step 9: the bandit, learning from samples**

**Core theory**

A bandit has K arms and no state. Each pull gives a noisy reward from a hidden distribution, and the goal is the highest total reward. So it's the Bellman problem with `T` and `R` hidden and the states removed.

Since you can't compute the value of an arm, you **estimate** it by averaging what you've seen. Store `N[a]`, the pulls of arm `a`, and `Q[a]`, its running average. After a new reward `r`:

```
Q[a] ← Q[a] + (r − Q[a]) / N[a]
```

This is the ordinary mean written as "move the estimate toward the new sample by 1/N of the gap". Remember this form: `estimate += step_size × (target − estimate)`. Q-learning and SARSA are this update with a different target.

The new difficulty is **exploration**. If you always pull the arm with the highest current `Q`, one lucky early reward can lock you onto a mediocre arm forever. There are two fixes:

- **ε-greedy:** with probability ε pull a random arm, otherwise the best-looking one.
- **Optimism:** start every `Q` at a high value, so every arm looks great until tried and gets a turn.



In [ ]:

K = 10

def run(epsilon, optimism=0.0, steps=1000, seed=0):
    rng = np.random.default_rng(seed)
    true_means = rng.normal(size=K)              # hidden from the agent
    Q = np.full(K, optimism, dtype=float)        # float, or updates get truncated
    N = np.zeros(K)                              # pulls per ARM, not total steps
    total = 0.0
    for _ in range(steps):
        a = rng.integers(K) if rng.random() < epsilon else Q.argmax()
        r = rng.normal(true_means[a])
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]                # running mean
        total += r
    return total / steps, true_means.max()

def avg(epsilon, optimism=0.0, n=200):
    out = np.array([run(epsilon, optimism, seed=s) for s in range(n)])
    return out.mean(0)

for eps, opt in [(0.0, 0.0), (0.1, 0.0), (0.0, 5.0)]:
    got, best = avg(eps, opt)
    print(f"eps={eps}  optimism={opt}:  avg reward {got:.3f}   (best arm on average: {best:.3f})")


eps=0.0  optimism=0.0:  avg reward 1.009   (best arm on average: 1.505)
eps=0.1  optimism=0.0:  avg reward 1.276   (best arm on average: 1.505)
eps=0.0  optimism=5.0:  avg reward 1.370   (best arm on average: 1.505)



Pure greedy does worst because it locks onto whatever arm looked good first. ε-greedy fixes some of that. Optimism with no random moves does best here, because it forces every arm to be tried early and then settles down.dy, plus the gradient and contextual variants in the same short step.

Step 10: Upper Confidence Bound (UCB), gradient bandit, contextual bandit

Core theory

All three keep the running-average update from step 9 or change one thing around it.

UCB changes how you explore. Rather than random moves, add a bonus to each arm that shrinks the more it has been pulled, and pick the arm with the highest Q + bonus. Arms you know little about get a big bonus, so they get tried. As N[a] grows, the bonus fades and the real estimate takes over.
Gradient bandit changes what you learn. Instead of estimating values, keep a preference H[a] per arm and choose arms by sampling from softmax(H). After each reward, raise the preference of the arm you pulled if the reward beat your average so far (the baseline), and lower it otherwise. Preferences for the other arms move the opposite way.
Contextual bandit adds a state, called the context, that you see before choosing. The best arm depends on it, but your action doesn't affect what context comes next. So the table just gets an extra index: Q[ctx, a].

In [ ]:
import numpy as np
K = 10

def run_ucb(c=2, steps=1000, seed=0):
    rng = np.random.default_rng(seed)
    true_means = rng.normal(size=K)                    # hidden from the agent
    Q, N, total = np.zeros(K), np.zeros(K), 0.0
    for t in range(1, steps + 1):                      # t starts at 1 so log(t) is not -inf
        a = np.argmax(Q + c * np.sqrt(np.log(t) / (N + 1e-6)))   # unpulled arms go first
        r = rng.normal(true_means[a])
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]
        total += r
    return total / steps, true_means.max()

res = np.array([run_ucb(seed=s) for s in range(200)]).mean(0)
print(f"UCB c=2         avg reward {res[0]:.3f}   (best arm on average: {res[1]:.3f})")

UCB c=2         avg reward 1.353   (best arm on average: 1.505)


In [ ]:
import numpy as np
K = 10

def run_grad(alpha=0.1, steps=1000, seed=0):
    rng = np.random.default_rng(seed)
    true_means = rng.normal(size=K)
    H, baseline, total = np.zeros(K), 0.0, 0.0
    for t in range(1, steps + 1):
        pi = np.exp(H - H.max()); pi /= pi.sum()       # softmax, max subtracted for stability
        a = rng.choice(K, p=pi)
        r = rng.normal(true_means[a])
        baseline += (r - baseline) / t                 # running mean of ALL rewards
        H += alpha * (r - baseline) * (np.eye(K)[a] - pi)
        total += r
    return total / steps, true_means.max()

res = np.array([run_grad(seed=s) for s in range(200)]).mean(0)
print(f"gradient a=0.1  avg reward {res[0]:.3f}   (best arm on average: {res[1]:.3f})")

gradient a=0.1  avg reward 1.316   (best arm on average: 1.505)


In [ ]:
import numpy as np

class ContextualBandit:
    def __init__(self, n_contexts, n_arms, seed=0):
        self.rng = np.random.default_rng(seed)
        self.means = self.rng.normal(size=(n_contexts, n_arms))   # hidden value of each (context, arm)
        self.n_contexts = n_contexts
    def reset(self):
        self.ctx = self.rng.integers(self.n_contexts)
        return self.ctx
    def step(self, a):
        r = self.rng.normal(self.means[self.ctx, a])
        self.ctx = self.rng.integers(self.n_contexts)             # next context ignores the action
        return self.ctx, r

env = ContextualBandit(5, 4)
rng = np.random.default_rng(1)
Q, N = np.zeros((5, 4)), np.zeros((5, 4))
ctx = env.reset()
for _ in range(20000):
    a = rng.integers(4) if rng.random() < 0.1 else Q[ctx].argmax()
    nxt, r = env.step(a)
    N[ctx, a] += 1
    Q[ctx, a] += (r - Q[ctx, a]) / N[ctx, a]
    ctx = nxt
print("contextual: Q shape", Q.shape, "| best arm per context correct:", (Q.argmax(1) == env.means.argmax(1)).all())

contextual: Q shape (5, 4) | best arm per context correct: True


Step 11: Q-learning

Core theory

In planning, you computed Q(s,a) by averaging over all the places an action could lead, using T and R. Now you don't have them, so you take one sample of what happened, (s, a, r, s'), and use it in place of the average:

target = r + γ · max_a' Q(s', a')

This is the Bellman lookahead from step 7, with two changes. The average over s' is replaced by the one s' you landed in. The V(s') is replaced by your current best guess, max_a' Q(s', a'), the value of playing the best move next. Then use the step 9 update:

Q(s,a) ← Q(s,a) + lr · (target − Q(s,a))

Nothing else is new. The step size is now a fixed lr instead of 1/N, so that older estimates get gradually replaced as Q improves.

Two facts about Q-learning are worth keeping:

It uses max in the target, so it learns the value of the best policy, even while the agent behaves with random ε-greedy moves. This is called off-policy learning.
The goal cell is never updated, so its Q stays 0 and the agent doesn't bootstrap past the end of the episode. Forgetting this is a classic bug.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def step(s, a):
    s_new = min(s + 1, 3) if a == 1 else max(s - 1, 0)
    return s_new, (1.0 if s_new == 3 else 0.0), s_new == 3

gamma, lr, eps = 0.9, 0.1, 0.1
Q = np.zeros((4, 2))                                 # row 3 is the goal cell, stays 0

def act(s):
    return rng.integers(2) if rng.random() < eps else Q[s].argmax()

for episode in range(2000):
    s, done = 0, False
    while not done:
        a = act(s)
        s_new, r, done = step(s, a)
        target = r + gamma * Q[s_new].max()          # the line that defines Q-learning
        Q[s, a] += lr * (target - Q[s, a])
        s = s_new

print(Q[:3].round(2))
print("greedy policy:", Q[:3].argmax(1))

[[0.73 0.81]
 [0.73 0.9 ]
 [0.81 1.  ]]
greedy policy: [1 1 1]


In [ ]:
Q

array([[0.72899739, 0.81      ],
       [0.72899858, 0.9       ],
       [0.8099977 , 1.        ],
       [0.        , 0.        ]])

SARSA uses the same update as Q-learning, but the target changes by one token:

Q-learning:  target = r + γ · max_a' Q(s', a')       # value of the BEST next move
SARSA:       target = r + γ · Q(s', a')              # value of the move you ACTUALLY take next

Here a' is the action your ε-greedy rule picks in s', and it's the same action you then execute on the next step. That has two consequences:

You must pick a' before the update, because the target needs it. So the loop chooses the first action before entering, then chooses the next action inside the loop, and carries it forward with s, a = s_new, a_new. Choosing it after the update is the standard bug.
It's on-policy. SARSA learns the value of the policy it is following, random exploratory moves included. Q-learning learns the value of the best policy, whatever it happens to do while exploring.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def step(s, a):
    s_new = min(s + 1, 3) if a == 1 else max(s - 1, 0)
    return s_new, (1.0 if s_new == 3 else 0.0), s_new == 3

gamma, lr, eps = 0.9, 0.1, 0.1
Q = np.zeros((4, 2))                                 # row 3 is the goal cell, stays 0

def act(s):
    return rng.integers(2) if rng.random() < eps else Q[s].argmax()

for episode in range(2000):
    s, done = 0, False
    a = act(s)                                       # pick the first action before the loop
    while not done:
        s_new, r, done = step(s, a)
        a_new = act(s_new)                           # pick the next action BEFORE the update
        target = r + gamma * Q[s_new, a_new]         # the only change from Q-learning
        Q[s, a] += lr * (target - Q[s, a])
        s, a = s_new, a_new                          # and this is the action you then take

print(Q[:3].round(2))
print("greedy policy:", Q[:3].argmax(1))

[[0.72 0.8 ]
 [0.69 0.88]
 [0.78 1.  ]]
greedy policy: [1 1 1]


Compare with Q-learning's table from step 11, which was [[0.73 0.81] [0.73 0.9] [0.81 1.0]]. SARSA's numbers are slightly lower: 0.88 against 0.9 for going right from cell 1, for example. That's the on-policy effect. SARSA's Q includes the cost of the 10% random moves it will make from the next cell on, while Q-learning pretends it will play perfectly. The greedy policy is the same [1 1 1] either way. In a world with a dangerous shortcut, this gap can change which policy each method prefers, but this corridor is too simple to show that.

Step 13: SARSA(λ)

Core theory

In SARSA, the surprise from one step, the TD error, updates only the one pair (s, a) you just took:

delta = r + γ·Q(s', a') − Q(s, a)

That is slow. A reward at the goal only reaches the cell next to it on the first pass, and the cell before that has to wait for a later episode. SARSA(λ) makes the surprise update every pair you visited recently, each in proportion to how recently. It keeps a table e[s, a] called the eligibility trace:

Visiting (s, a) adds 1 to e[s, a].
Update every entry: Q += lr · delta · e.
Fade the traces: e *= γ · λ.
Reset e to zero at the start of every episode.

The parameter λ between 0 and 1 controls how far back the credit reaches. With λ = 0, the traces vanish after each step, so only the latest pair updates and you get plain SARSA. With λ near 1, credit reaches far back along the path.

In [ ]:
import numpy as np

def step(s, a):
    s_new = min(s + 1, 3) if a == 1 else max(s - 1, 0)
    return s_new, (1.0 if s_new == 3 else 0.0), s_new == 3

def train(lam, episodes=10, gamma=0.9, lr=0.1, eps=0.1, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros((4, 2))
    act = lambda s: rng.integers(2) if rng.random() < eps else Q[s].argmax()
    for _ in range(episodes):
        e = np.zeros_like(Q)                          # reset traces every episode
        s, done = 0, False
        a = act(s)
        while not done:
            s_new, r, done = step(s, a)
            a_new = act(s_new)
            delta = r + gamma * Q[s_new, a_new] - Q[s, a]
            e[s, a] += 1
            Q += lr * delta * e                       # every pair moves, weighted by its trace
            e *= gamma * lam
            s, a = s_new, a_new
    return Q

for lam in (0.0, 0.9):
    Qs = np.mean([train(lam, seed=k)[:3, 1] for k in range(200)], axis=0)
    print(f"lambda={lam}:  Q(right) after 10 episodes {Qs.round(2)}")
print("converged SARSA Q(right), for reference   [0.8  0.88 1.  ]")

lambda=0.0:  Q(right) after 10 episodes [0.06 0.23 0.65]
lambda=0.9:  Q(right) after 10 episodes [0.47 0.56 0.65]
converged SARSA Q(right), for reference   [0.8  0.88 1.  ]


This averages 200 runs, and it measures how much each method has learned after only 10 episodes. Cell 2, next to the goal, is at 0.65 for both, because it gets its reward on the very step that produces it. The difference is in cells 0 and 1: λ = 0 has barely moved cell 0 (0.06), while λ = 0.9 has pushed the credit back along the whole path (0.47). Traces get the same information further down the chain per episode.

Two things to remember for the exercise:

Reset e at the start of every episode, or credit leaks across episodes.
Q += lr * delta * e updates the whole table at once, so it needs no loop. The trace table e has the same shape as Q, which is the shape check to write in the comment.name for thi noteb

Q1–Q2 (Current Phase): Lock in Tabular RL $\rightarrow$ Deep RL (DQN/PPO) $\rightarrow$ Transition into SFT & LoRA Fine-Tuning.Q3: Focus on Alignment Methods (DPO, GRPO) and AI Evaluation framework design.Q4: Synthesize everything into Autonomous Agentic Architectures, backed by Mechanistic Interpretability diagnostics.